[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# COPY


## What you will be able to do

Load many rows with `COPY` rather than with an `INSERT` for each, in both drivers, and say roughly
how much that is worth. Write rows from Python objects and from a file. Read a whole table or query
back out the same way. Use the binary format and know what it asks of you in return. And read the
error a malformed row produces, which stops the whole load rather than skipping the row.


## The idea

### The problem

`executemany` in **Placeholders and Identifiers** wrote rows in one call, and psycopg 3 already sends
that batch through pipeline mode, so it is not one round trip per row. It is still one `INSERT` per
row as far as the server is concerned: parsed, planned, and run, fifty thousand times.

`COPY` is a different statement. One of them loads the whole batch, the rows go over in a stream, and
the server writes them without planning anything per row. That is where the difference comes from,
and it is large enough that the choice matters for anything above a few thousand rows.

### What COPY is

A PostgreSQL statement that moves rows between a table and a stream. `COPY ... FROM STDIN` reads them
from the client, `COPY ... TO STDOUT` writes them to the client, and both drivers give you a way to
be that client.

`COPY ... FROM '/path/to/file'` also exists and this guide never uses it, because it reads a file on
the **server** and needs superuser rights. Everything here is the STDIN form, which reads from your
process.

### Why it works that way

An `INSERT` is a statement, so every one of them is parsed and planned. `COPY` is one statement whose
payload is rows, so the parsing and planning happen once. The rows themselves still have to be
parsed from text or binary, and still go through the same table machinery, which is why the gap is
large and not infinite.

### Where this shows up

Loading a file somebody sent you, seeding a database, moving a table between servers, and the export
half of any reporting job. It is also the answer to the **Server-Side Cursors** notebook's export,
which read rows one batch at a time and wrote them by hand.

### What this notebook covers

`COPY` in, from rows and from a file. `COPY` out. The binary format. asyncpg's three methods. What
each of them costs against `executemany`. Then the four failures, of which the interesting one is a
row with an extra column.

Timings here are bands rather than numbers. A local Unix socket is the least favorable place to
measure this, because the round trip `COPY` saves is almost free, so what you would see over a
network is a wider gap than what is printed below.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import time

import psycopg

rows = [(n, "click", n % 7) for n in range(1, 50_001)]


def fresh():
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS loaded")
        conn.execute("CREATE TABLE loaded (id int, kind text, size int)")


def timed(load):
    fresh()
    start = time.perf_counter()
    load()
    return time.perf_counter() - start


def with_executemany():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        cur.executemany("INSERT INTO loaded VALUES (%s, %s, %s)", rows)


def with_copy():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
            for row in rows:
                copy.write_row(row)


slow, fast = timed(with_executemany), timed(with_copy)

print("50,000 rows, one local socket")
print("  executemany: the baseline")
print("  COPY:       ", "an order of magnitude faster" if slow / fast > 10 else "faster")
```

```
50,000 rows, one local socket
  executemany: the baseline
  COPY:        an order of magnitude faster
```

The two loops are the same shape and write the same rows. One sends fifty thousand statements and
the other sends one, and that is the whole of the difference.


## Setup

Twelve imports, both drivers, the server, and fifty thousand rows waiting to be written.

- `psycopg` and `asyncpg` are the drivers, and `errors` is the exception classes
- `io` and `Path` hold a copy in memory and a copy on disk, which are the two sources a load has
- `time` measures, `tempfile` makes somewhere to write, and `subprocess`, `sys`, `os`, `getpass`
  stand the server up with `version` and `PackageNotFoundError`

`ROWS` is what every load in this notebook writes, `fresh` empties the table so each one starts the
same way, and `against` turns a timing into a band, because a number measured on a shared machine is
not the same twice.


In [1]:
import getpass
import io
import os
import subprocess
import sys
import tempfile
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import asyncpg
import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

ROWS = [(n, "click", n % 7) for n in range(1, 50_001)]              # what every load below writes
WORK = Path(tempfile.mkdtemp(prefix="copy-"))


def fresh():
    """An empty table, so each way of loading starts from the same place."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS loaded")
        conn.execute("CREATE TABLE loaded (id int, kind text, size int)")


def loaded():
    """How many rows are in it now."""
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM loaded").fetchone()[0]


def timed(load):
    """Seconds taken to run a load against an empty table."""
    fresh()
    start = time.perf_counter()
    load()
    return time.perf_counter() - start


def against(baseline, measured):
    """How much faster, as a band rather than a number.

    A timing on a shared machine is not repeatable to a digit, so this notebook reports which
    band a result fell in. The bands are far enough apart that the answer is the same on every
    run, which a printed ratio was not.
    """
    ratio = baseline / measured
    if ratio < 2:
        return "about the same"
    if ratio <= 10:
        return "several times faster"
    return "an order of magnitude faster"


print("server:", start_server())
print(report())
fresh()
print("loaded is empty:", loaded(), "| rows to write:", len(ROWS))


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
loaded is empty: 0 | rows to write: 50000


## Worked examples

### COPY in, from rows you have

`cursor.copy(...)` opens the stream, and `write_row` puts a Python tuple into it:


In [2]:
fresh()

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
        for row in ROWS[:1000]:
            copy.write_row(row)

print("rows:", loaded())


rows: 1000


The `with` block is the statement: it opens the `COPY`, streams whatever you write into it, and
closes it at the end. If the block is left by an exception the whole load is rolled back, which is
the last of this notebook's failures.

`write_row` takes Python objects and adapts them the way parameters are adapted, which is the same
machinery **Types and Adaptation** described.

### COPY in, from a file

A file that is already in the right shape does not need to become Python objects at all:


In [3]:
csv = WORK / "rows.csv"
csv.write_text("".join(f"{n},click,{n % 7}\n" for n in range(1, 1001)))

fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
        copy.write(csv.read_text())

print("rows:", loaded(), "| from a file of", csv.stat().st_size, "bytes")


rows: 1000 | from a file of 11893 bytes


`write` takes text or bytes and sends them as they are, which is faster than `write_row` when the
data is already formatted and is the only sensible way to load a file you were given.

Reading it in blocks rather than all at once is the version for a file too big to hold:


In [4]:
fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
        with open(csv) as lines:
            while block := lines.read(8192):
                copy.write(block)

print("rows:", loaded())


rows: 1000


### COPY out

The same statement the other way. `COPY ... TO STDOUT` streams rows to you, and the cursor is
iterable over the blocks as they arrive:


In [5]:
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    blocks = []
    with cur.copy("COPY (SELECT id, kind FROM loaded ORDER BY id LIMIT 3) TO STDOUT") as copy:
        for block in copy:
            blocks.append(bytes(block).decode())

print("what came out:", repr("".join(blocks)))


what came out: '1\tclick\n2\tclick\n3\tclick\n'


Tab separated, one row per line, which is `COPY`'s text format. Any query can be the source, not just
a table, which makes this a way to export exactly what you want.

Writing it to a file is the same loop with somewhere to put the blocks:


In [6]:
export = WORK / "export.csv"

with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with open(export, "wb") as out:
        with cur.copy("COPY loaded TO STDOUT WITH (FORMAT csv, HEADER)") as copy:
            for block in copy:
                out.write(block)

print("first two lines:", export.read_text().splitlines()[:2])
print("lines in all:   ", len(export.read_text().splitlines()))


first two lines: ['id,kind,size', '1,click,1']
lines in all:    1001


### The binary format

Binary skips turning every value into text and back. It is faster, and it asks you to say what the
columns are, because there is no text for the server to infer from:


In [7]:
fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT binary)") as copy:
        copy.set_types(["int4", "text", "int4"])
        for row in ROWS[:1000]:
            copy.write_row(row)

print("rows:", loaded())


rows: 1000


Leaving out `set_types` is the third of the Common errors. Binary is also less forgiving about what
it will accept: a text load will take anything the input type can be parsed from, and a binary load
wants exactly the type you declared.

### What each one costs

Four ways of writing fifty thousand rows, against `executemany` as the baseline:


In [8]:
def by_executemany():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        cur.executemany("INSERT INTO loaded VALUES (%s, %s, %s)", ROWS)


def by_copy_text():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
            for row in ROWS:
                copy.write_row(row)


def by_copy_binary():
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT binary)") as copy:
            copy.set_types(["int4", "text", "int4"])
            for row in ROWS:
                copy.write_row(row)


baseline = timed(by_executemany)
for name, load in (("COPY, text", by_copy_text), ("COPY, binary", by_copy_binary)):
    print(f"  {name:<14} {against(baseline, timed(load))}")
print(f"  {'executemany':<14} the baseline, and already pipelined")


  COPY, text     an order of magnitude faster
  COPY, binary   an order of magnitude faster
  executemany    the baseline, and already pipelined


Both `COPY` forms are in the same band, and on a local socket the difference between text and binary
is too small to separate honestly. Binary wins by more as rows get wider and as the network gets
slower, which is exactly the case this machine cannot show you.

`executemany` being "already pipelined" is worth keeping in mind: psycopg 3 does not send one round
trip per row, so what `COPY` is saving here is the per-row parse and plan, not the waiting.

### asyncpg

Three methods, one for each source, and all of them are `COPY` underneath:


In [9]:
fresh()
conn = await asyncpg.connect(database="guide")

status = await conn.copy_records_to_table("loaded", records=ROWS[:1000],
                                          columns=["id", "kind", "size"])
print("copy_records_to_table ->", repr(status), "| rows:", loaded())

sink = io.BytesIO()
out = await conn.copy_from_query("SELECT id, kind FROM loaded ORDER BY id LIMIT 3", output=sink)
print("copy_from_query       ->", repr(out), "|", sink.getvalue())


copy_records_to_table -> 'COPY 1000' | rows: 1000
copy_from_query       -> 'COPY 3' | b'1\tclick\n2\tclick\n3\tclick\n'


`copy_records_to_table` takes an iterable of tuples, `copy_to_table` takes a file, and
`copy_from_query` writes a query's rows to a file or anything with a `write`. The return value is the
command tag, which is how many rows moved.

The one place asyncpg is less helpful is a record of the wrong width:


In [10]:
try:
    await conn.copy_records_to_table("loaded", records=[(1, "click")],
                                     columns=["id", "kind", "size"])
except (IndexError, asyncpg.exceptions.PostgresError) as error:
    print("a record with two values for three columns:", type(error).__name__ + ":", error)

await conn.close()


a record with two values for three columns: IndexError: tuple index out of range


That is an `IndexError` from asyncpg's own encoder rather than anything the server said, because the
record ran out before the columns did. Checking the width of your records before handing them over is
worth doing, since this message does not say which record.

### When to reach for which

| What you are doing | What to use |
|---|---|
| a handful of rows | `execute` or `executemany` |
| many rows, from Python objects | `cur.copy(...)` and `write_row` |
| many rows, from a file already in shape | `cur.copy(...)` and `write` |
| many rows, and speed matters most | binary `COPY` with `set_types` |
| many rows, in asyncpg | `copy_records_to_table`, or `copy_to_table` for a file |
| a table or query out | `COPY ... TO STDOUT`, iterating the cursor |
| the same in asyncpg | `copy_from_query` or `copy_from_table` |
| rows that need checking first | `INSERT`, because `COPY` is all or nothing |

`COPY` is the default for a bulk load. Reach for `executemany` when the rows need per-row error
handling, because a `COPY` that hits a bad row takes the whole load down with it, which is the point
of the last section.

### A load from a file, finished

Everything above, as the job it is for: a file arrives, it is loaded in one statement, and the
result is reported.


In [11]:
def load_file(path, table="loaded"):
    """Load a CSV in one statement, and say how many rows landed."""
    fresh()
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy(f"COPY {table} (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
            with open(path) as lines:
                while block := lines.read(65536):
                    copy.write(block)
        return conn.execute(f"SELECT count(*) FROM {table}").fetchone()[0]


big = WORK / "big.csv"
big.write_text("".join(f"{n},click,{n % 7}\n" for n in range(1, 50_001)))

print("file:", big.stat().st_size, "bytes")
print("loaded:", load_file(big), "rows")


file: 688894 bytes
loaded: 50000 rows


The table name is formatted into the statement, which **Placeholders and Identifiers** said to be
careful about: it is safe here because the name is this function's own default and not anything a
caller supplied, and the moment it were, `sql.Identifier` would be the way to write it.

### Where each part came from

| In the load | What it relies on | The section that showed it |
|---|---|---|
| `cur.copy("COPY ... FROM STDIN")` | one statement for the whole batch | COPY in, from rows |
| `copy.write(block)` | text sent as it is, without becoming objects | COPY in, from a file |
| reading the file in blocks | a file bigger than memory | COPY in, from a file |
| the `with` block around the copy | a load that is all or nothing | COPY in, from rows |
| counting afterwards | the table, asked rather than assumed | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/08-copy-solutions.ipynb).

**1.** Load a thousand rows with `COPY` from Python tuples, and count them.


In [12]:
# your code here


**2.** Write those rows out to a file with a header, and print the first two lines.


In [13]:
# your code here


**3.** Load the same rows in binary, saying what the column types are.


In [14]:
# your code here


**4.** Time `executemany` against `COPY` for the same rows and print which band the difference falls
in.


In [15]:
# your code here


**5.** Load a thousand rows through asyncpg and print what the call returned.


In [16]:
# your code here


**6.** Load a CSV whose third line has an extra column, and show how many rows are in the table
afterwards.


In [17]:
# your code here


## Common errors

### psycopg.errors.BadCopyFileFormat: extra data after last expected column


In [18]:
broken = WORK / "broken.csv"
broken.write_text("1,click,1\n2,view,2\n3,a,name,with,commas,3\n4,view,4\n")

fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
        copy.write(broken.read_text())


BadCopyFileFormat: extra data after last expected column
CONTEXT:  COPY loaded, line 3: "3,a,name,with,commas,3"

The third line has six fields where the table has three columns, because a value contained commas
and nothing quoted it. This is the most common thing wrong with a file somebody sends you.

What matters as much as the message is what it left behind:


In [19]:
print("rows in the table after that failure:", loaded())


rows in the table after that failure: 0


None. `COPY` is one statement, so it is all or nothing: the two good rows before the bad one are
gone too. That is usually what you want for a load, and it means a file has to be clean before it is
worth starting.

Quoting the field is what the file should have done, and `COPY` understands it:


In [20]:
fixed = WORK / "fixed.csv"
fixed.write_text('1,click,1\n2,view,2\n3,"a,name,with,commas",3\n4,view,4\n')

fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT csv)") as copy:
        copy.write(fixed.read_text())

with psycopg.connect("dbname=guide") as conn:
    print("rows:", loaded(), "| the third one:",
          conn.execute("SELECT kind FROM loaded WHERE id = 3").fetchone()[0])


rows: 4 | the third one: a,name,with,commas


### psycopg.ProgrammingError: COPY cannot be used with this method; use copy() instead


In [21]:
conn = psycopg.connect("dbname=guide")
try:
    conn.execute("COPY loaded FROM STDIN")
except psycopg.ProgrammingError as error:
    print(type(error).__module__ + "." + type(error).__name__ + ":", error)
finally:
    conn.close()                                                    # this one cannot be used again


psycopg.ProgrammingError: COPY cannot be used with this method; use copy() instead


`COPY ... FROM STDIN` is not a statement that can be run and then be finished with: the server
expects rows to follow on the same connection. `execute` has nowhere to put them, so psycopg
refuses rather than leaving the connection half way through a statement.

The connection is closed by hand above rather than left to a `with` block, because after this
refusal it is stuck part way through a statement and tidying it up prints a warning of its own. A
fresh one, and the right method:


In [22]:
fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
        copy.write_row((1, "click", 1))

print("rows:", loaded())


rows: 1


### psycopg.errors.ProtocolViolation: insufficient data left in message


In [23]:
fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT binary)") as copy:
        for row in ROWS[:5]:
            copy.write_row(row)                                     # no set_types above


ProtocolViolation: insufficient data left in message
CONTEXT:  COPY loaded, line 1, column id

A binary `COPY` carries no type information of its own, so the values have to be written in exactly
the form the columns expect. Without `set_types` psycopg guesses from the Python objects, the bytes
do not line up with what the columns are, and the server stops with a complaint about the message
rather than about the types, which is as close as it can get to naming the real problem.

`set_types` is the whole fix, and the list is the columns in the order the `COPY` named them:


In [24]:
fresh()
with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
    with cur.copy("COPY loaded (id, kind, size) FROM STDIN WITH (FORMAT binary)") as copy:
        copy.set_types(["int4", "text", "int4"])
        for row in ROWS[:5]:
            copy.write_row(row)

print("rows:", loaded())


rows: 5


### No error, and a load that vanished: an exception inside the copy block


In [25]:
fresh()
try:
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
            for number, row in enumerate(ROWS[:1000]):
                if number == 500:
                    raise RuntimeError("something went wrong half way through")
                copy.write_row(row)
except RuntimeError as error:
    print("the loop stopped at:", error)

print("rows in the table:", loaded())


the loop stopped at: something went wrong half way through
rows in the table: 0


Five hundred rows were written into the stream and none of them are there. The `with` block ends the
`COPY` by telling the server to discard it when an exception leaves, which is the right thing and is
worth knowing before you build a job on top of it.

It means a partial load is not a state you can get into by accident, and it also means there is no
resuming: a load that fails at row nine hundred thousand starts again at row one. Splitting a very
large file into batches, each its own `COPY`, is how that is handled:


In [26]:
def load_in_batches(rows, batch=250):
    """Several COPY statements, so a failure costs one batch rather than the file."""
    written = 0
    with psycopg.connect("dbname=guide") as conn, conn.cursor() as cur:
        for start in range(0, len(rows), batch):
            with cur.copy("COPY loaded (id, kind, size) FROM STDIN") as copy:
                for row in rows[start:start + batch]:
                    copy.write_row(row)
            conn.commit()                                           # this batch is safe now
            written += len(rows[start:start + batch])
    return written


fresh()
print("written in batches:", load_in_batches(ROWS[:1000]), "| in the table:", loaded())


written in batches: 1000 | in the table: 1000


## Recap

- `COPY` is one statement that carries many rows, where `executemany` is many statements. On a local
  socket that is an order of magnitude for fifty thousand rows, and more over a network.
- `cur.copy(...)` opens the stream. `write_row` takes Python objects and `write` takes text or bytes
  already in shape, which is the faster one for a file.
- `COPY ... TO STDOUT` reads a table or any query back out, and the cursor is iterable over the
  blocks.
- The binary format is faster and needs `set_types`, because it carries no type information.
- asyncpg has `copy_records_to_table`, `copy_to_table`, `copy_from_query` and `copy_from_table`, all
  of them `COPY` underneath.
- A `COPY` is all or nothing. A bad row, or an exception in the block, discards the whole load, so
  large files are loaded in batches if resuming matters.
- `COPY ... FROM '/path'` reads a file on the server and needs superuser rights, which is why this
  guide never uses it.


## What is next

The **Pipeline Mode** notebook is about the waiting rather than the work: sending many statements
without waiting for each answer, what that saves, and the error that is reported against a statement
which was perfectly fine.


---

&#8592; **Previous:** [Server-Side Cursors](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/07-server-side-cursors.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Pipeline Mode](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/09-pipeline-mode.ipynb) &#8594;
